In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, recall_score, confusion_matrix

In [2]:
df = pd.read_excel("data/raw/Aggregated_Flood_Data.xlsx", sheet_name="Aggregated")

df = df[df["Year"].between(1900, 2026)].dropna(subset=["Fatalities"])

numeric_features = ["Year", "Lat", "Lon", "Affected", "Injured"]
categorical_features = ["Country"]

X = df[numeric_features + categorical_features]
y = (df["Fatalities"] >= 1).astype(int)

X.shape, y.value_counts(normalize=True)

((129721, 6),
 Fatalities
 0    0.82837
 1    0.17163
 Name: proportion, dtype: float64)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=522
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(103776, 6) (25945, 6)
0.1716 0.1716


In [4]:
preprocessor = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical_features),
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

X_train_proc.shape, X_test_proc.shape

((103776, 376), (25945, 376))

## Model Selection

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=522)
scoring = ["f1", "recall"]

In [6]:
dummy_pipe = make_pipeline(preprocessor, DummyClassifier(strategy="most_frequent"))
dummy_scores = cross_validate(dummy_pipe, X_train, y_train, cv=cv, scoring=scoring)

results = {"Dummy": {m: dummy_scores[f"test_{m}"].mean() for m in scoring}}
pd.DataFrame(results).T.round(4)

,f1,recall
Dummy,0.0,0.0


In [7]:
models = {
    "Logistic Regression": make_pipeline(preprocessor, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=522)),
    "Decision Tree": make_pipeline(preprocessor, DecisionTreeClassifier(max_depth=15, class_weight="balanced", random_state=522)),
}

for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

In [8]:
cv_results = pd.DataFrame(results).T.sort_values("f1", ascending=False).round(4)
cv_results

,f1,recall
Decision Tree,0.6078,0.7055
Logistic Regression,0.5554,0.6956
Dummy,0.0000,0.0000


## Hyperparameter tuning and final test set evaluation

In [9]:
dt_pipe = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(class_weight="balanced", random_state=522),
)

param_dist = {
    "decisiontreeclassifier__max_depth": [5, 10, 15, 20, 30, None],
    "decisiontreeclassifier__min_samples_leaf": [1, 5, 10, 20],
}

search = RandomizedSearchCV(dt_pipe, param_dist, n_iter=10, cv=5, scoring="f1", random_state=522)
search.fit(X_train, y_train)

best_model = search.best_estimator_
search.best_params_, round(search.best_score_, 4)

({'decisiontreeclassifier__min_samples_leaf': 5,
  'decisiontreeclassifier__max_depth': 20},
 np.float64(0.6121))

In [10]:
y_pred = best_model.predict(X_test)

test_metrics = pd.Series({
    "f1": f1_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
}).round(4)

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual Non-Fatal", "Actual Fatal"],
    columns=["Pred Non-Fatal", "Pred Fatal"],
)

print(test_metrics)
cm

f1        0.6114
recall    0.7276
dtype: float64


,Pred Non-Fatal,Pred Fatal
Actual Non-Fatal,18587,2905
Actual Fatal,1213,3240
